In [1]:
!git clone https://github.com/choi403/DiffusionGuard.git

Cloning into 'DiffusionGuard'...
remote: Enumerating objects: 37, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (36/36), done.
remote: Total 37 (delta 12), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (37/37), 1.60 MiB | 4.99 MiB/s, done.
Resolving deltas: 100% (12/12), done.


In [11]:
!pip install -r requirements.txt 

In [ ]:
!pip install opencv-python

In [ ]:
!pip install hydra-core --upgrade

In [12]:
!python main.py --config-name diffusionguard orig_image_name='keanu.png' mask_image_names='["keanu_mask.png"]'

exp_name: exp
method: diffusionguard
orig_image_name: keanu.png
mask_image_names:
- keanu_mask.png
model:
  inpainting: runwayml/stable-diffusion-inpainting
training:
  size: 512
  iters: 800
  grad_reps: 1
  batch_size: 1
  eps: 0.06274509803921569
  step_size: 0.00392156862745098
  num_inference_steps: 4
  mask:
    generation_method: contour_shrink
    contour_strength: 1.1
    contour_iters: 15
    contour_smoothness: 0.1

[2026-02-08 13:42:46,963][httpx][INFO] - HTTP Request: GET https://huggingface.co/api/models/runwayml/stable-diffusion-inpainting "HTTP/1.1 307 Temporary Redirect"
[2026-02-08 13:42:47,046][httpx][INFO] - HTTP Request: GET https://huggingface.co/api/models/stable-diffusion-v1-5/stable-diffusion-inpainting "HTTP/1.1 200 OK"
[2026-02-08 13:42:47,140][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/runwayml/stable-diffusion-inpainting/resolve/main/model_index.json "HTTP/1.1 307 Temporary Redirect"
[2026-02-08 13:42:47,221][httpx][INFO] - HTTP Request: HEAD 

In [ ]:
import torch
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image
import matplotlib.pyplot as plt
import os
from PIL import Image, ImageOps # Added ImageOps for easy inversion

# --- Configuration ---
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = "runwayml/stable-diffusion-inpainting"

# Paths (Based on your previous main.py run)
ORIGINAL_PATH = "assets/keanu.png"
MASK_PATH = "assets/keanu_mask.png"
PROTECTED_PATH = "protected_images/exp/keanu/adv_image.png" # Output from main.py

# Edit Settings
PROMPT = "chef cooking in a restaurant" # The edit we want to force
GUIDANCE_SCALE = 7.5
STEPS = 50
SEED = 42

# --- 1. Load Pipeline ---
print(f"Loading Inpainting Model: {MODEL_ID}...")
pipe = StableDiffusionInpaintPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if DEVICE == 'cuda' else torch.float32,
    safety_checker=None
).to(DEVICE)

# --- 2. Load Images ---
def load_image(path):
    return Image.open(path).convert("RGB").resize((512, 512))

def load_mask(path):
    # Mask must be grayscale (L)
    return Image.open(path).convert("L").resize((512, 512))

# --- 2. Load & Invert Mask ---
def load_and_invert_mask(path):
    # Load as grayscale
    mask = Image.open(path).convert("L").resize((512, 512))
    # INVERT: Makes White -> Black and Black -> White
    # Now, the area originally marked in the file will be the TARGET for editing
    inverted_mask = ImageOps.invert(mask)
    return inverted_mask

print("Loading images...")
if not os.path.exists(PROTECTED_PATH):
    raise FileNotFoundError(f"Protected image not found at {PROTECTED_PATH}. Did main.py finish successfully?")

img_orig = load_image(ORIGINAL_PATH)
img_prot = load_image(PROTECTED_PATH)
mask = load_and_invert_mask(MASK_PATH)

# --- 3. Run Inpainting (The Attack) ---
generator = torch.Generator(DEVICE).manual_seed(SEED)

print(f"Inpainting Original Image with prompt: '{PROMPT}'...")
result_orig = pipe(
    prompt=PROMPT,
    image=img_orig,
    mask_image=mask,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator
).images[0]

print(f"Inpainting Protected Image with prompt: '{PROMPT}'...")
# Reset generator for fair comparison
generator = torch.Generator(DEVICE).manual_seed(SEED) 
result_prot = pipe(
    prompt=PROMPT,
    image=img_prot,
    mask_image=mask,
    num_inference_steps=STEPS,
    guidance_scale=GUIDANCE_SCALE,
    generator=generator
).images[0]

# --- 4. Visualize Results ---
fig, ax = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Original Flow
ax[0, 0].imshow(img_orig)
ax[0, 0].set_title("Original Image")
ax[0, 1].imshow(mask, cmap='gray')
ax[0, 1].set_title("Mask")
ax[0, 2].imshow(result_orig)
ax[0, 2].set_title(f"Result (Original)\nShould match prompt: '{PROMPT}'")

# Row 2: Protected Flow
ax[1, 0].imshow(img_prot)
ax[1, 0].set_title("Protected Image (DiffusionGuard)")
ax[1, 1].imshow(mask, cmap='gray')
ax[1, 1].set_title("Mask")
ax[1, 2].imshow(result_prot)
ax[1, 2].set_title(f"Result (Protected)\nShould FAIL to match prompt")

for a in ax.flat:
    a.axis('off')

plt.tight_layout()
plt.show()

# Save results for closer inspection
result_orig.save("eval_result_original.png")
result_prot.save("eval_result_protected.png")
print("Evaluation complete. Images saved.")

Loading Inpainting Model: runwayml/stable-diffusion-inpainting...


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

An error occurred while trying to fetch /home/jovyan/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /home/jovyan/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /home/jovyan/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /home/jovyan/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /home/jovyan/.cache/huggingface/hub/models--runwayml--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
You have disabled the safety checker for <class 'diffusers.pipelines.stable_diffusion.pipeline_stable_diffusion_inpaint.StableDiffusionInpaintPipeline'> by passing `safety_checker=None`. Ensure that you abide to the conditions of the Stable Diffusion license and do not expose unfiltered results in services or applications open to the public. Both the diffusers team and Hugging Face strongly recommend to keep the safety filter enabled in all public facing circumstances, disabling it only for use-cases that involve analyzing network be

Loading images...
Inpainting Original Image with prompt: 'chef cooking in a restaurant'...


  0%|          | 0/50 [00:00<?, ?it/s]

Inpainting Protected Image with prompt: 'chef cooking in a restaurant'...


  0%|          | 0/50 [00:00<?, ?it/s]